In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
!pip -q install rasterio scikit-image

## Upload Devanahalli files

In [ ]:
import os
from google.colab import files
os.makedirs("dev", exist_ok=True)
for name, data in files.upload().items():
    open(f"dev/{name}", "wb").write(data)
print("dev/:", sorted(os.listdir("dev")))

## Upload ORR files

In [ ]:
os.makedirs("orr", exist_ok=True)
for name, data in files.upload().items():
    open(f"orr/{name}", "wb").write(data)
for f in os.listdir("orr"):
    clean = f.replace(" (1)", "")
    if clean != f:
        os.rename(f"orr/{f}", f"orr/{clean}")
print("orr/:", sorted(os.listdir("orr")))

## Build 7-channel change patches (image_earlier + image_later + past-year mask)

In [ ]:
import numpy as np
import rasterio

PATCH, STRIDE, VAL_FRAC = 256, 128, 0.28
os.makedirs("cd/train", exist_ok=True)
os.makedirs("cd/val", exist_ok=True)
PAIRS = [(2017, 2024), (2017, 2019), (2019, 2024)]


def stretch(a):
    out = np.empty_like(a, "float32")
    for b in range(a.shape[0]):
        finite = a[b][np.isfinite(a[b])]
        lo, hi = np.percentile(finite, 2), np.percentile(finite, 98)
        out[b] = np.clip((a[b] - lo) / (hi - lo + 1e-6), 0, 1)
    return np.nan_to_num(out)


def rgb(aoi, year):
    with rasterio.open(f"{aoi}/stack_{year}.tif") as s:
        return stretch(s.read([3, 2, 1]).astype("float32")), s.height, s.width


def osm(aoi, year):
    return (rasterio.open(f"{aoi}/osm_roads_{year}_mask.tif").read(1) == 1).astype("uint8")


def tile(img0, img1, past, change, H, W, lo, hi, sub, tag, counts):
    stacked = np.concatenate([img0, img1, past[None].astype("float32")], 0)
    for y in range(0, H - PATCH + 1, STRIDE):
        for x in range(0, W - PATCH + 1, STRIDE):
            if x < lo or x + PATCH > hi:
                continue
            patch = stacked[:, y:y+PATCH, x:x+PATCH]
            if patch[:3].max() < 1e-4:
                continue
            np.savez_compressed(f"cd/{sub}/{tag}_{y}_{x}.npz",
                                img=patch.astype("float32"), mask=change[y:y+PATCH, x:x+PATCH])
            counts[sub] += 1


counts = {"train": 0, "val": 0}


def build(aoi, region):
    for y0, y1 in PAIRS:
        img0, H, W = rgb(aoi, y0)
        img1, _, _ = rgb(aoi, y1)
        o0, o1 = osm(aoi, y0), osm(aoi, y1)
        h = min(img0.shape[1], img1.shape[1], o0.shape[0], o1.shape[0])
        w = min(img0.shape[2], img1.shape[2], o0.shape[1], o1.shape[1])
        img0, img1, o0, o1 = img0[:, :h, :w], img1[:, :h, :w], o0[:h, :w], o1[:h, :w]
        change = (o1 & ~o0).astype("uint8")
        split = int(w * (1 - VAL_FRAC))
        if region == "val" and (y0, y1) == (2017, 2024):
            tile(img0, img1, o0, change, h, w, split, w, "val", "dev", counts)
        if region == "left":
            tile(img0, img1, o0, change, h, w, 0, split, "train", f"{aoi}{y0}{y1}", counts)
        if region == "full":
            tile(img0, img1, o0, change, h, w, 0, w, "train", f"{aoi}{y0}{y1}", counts)


build("dev", "val")
build("dev", "left")
build("orr", "full")
print("change patches:", counts)

## Model, loss and evaluation

In [ ]:
import glob
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=7, base=64):
        super().__init__()
        self.e1 = DoubleConv(in_ch, base)
        self.e2 = DoubleConv(base, base * 2)
        self.e3 = DoubleConv(base * 2, base * 4)
        self.e4 = DoubleConv(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.bott = DoubleConv(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2)
        self.d4 = DoubleConv(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = DoubleConv(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = DoubleConv(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.bott(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.head(d1)


class ChangeDS(Dataset):
    def __init__(self, folder, augment=False, synthetic=False):
        self.files = sorted(glob.glob(f"{folder}/*.npz"))
        self.augment = augment
        self.synthetic = synthetic

    def __len__(self):
        return len(self.files)

    def _synthetic(self, img, mask):
        past = img[6]
        ys, xs = np.where(past > 0.5)
        if len(ys) > 20:
            cy, cx = ys[np.random.randint(len(ys))], xs[np.random.randint(len(xs))]
            r = np.random.randint(20, 60)
            y0, y1 = max(0, cy - r), min(past.shape[0], cy + r)
            x0, x1 = max(0, cx - r), min(past.shape[1], cx + r)
            region = np.zeros_like(past, bool)
            region[y0:y1, x0:x1] = True
            roads = region & (past > 0.5)
            img[6][roads] = 0.0
            mask = mask.copy()
            mask[roads] = 1
        return img, mask

    def __getitem__(self, i):
        data = np.load(self.files[i])
        img, mask = data["img"].copy(), data["mask"].copy()
        if self.synthetic and np.random.rand() < 0.5:
            img, mask = self._synthetic(img, mask)
        if self.augment:
            if np.random.rand() < 0.5:
                img, mask = img[:, :, ::-1], mask[:, ::-1]
            if np.random.rand() < 0.5:
                img, mask = img[:, ::-1, :], mask[::-1, :]
            k = np.random.randint(4)
            if k:
                img, mask = np.rot90(img, k, (1, 2)), np.rot90(mask, k)
        return (torch.from_numpy(np.ascontiguousarray(img).astype("float32")),
                torch.from_numpy(np.ascontiguousarray(mask).astype("float32"))[None])


POS_WEIGHT = torch.tensor([10.0]).to(DEVICE)


def loss_fn(logits, target, alpha=0.3, beta=0.7, gamma=0.75, eps=1.0):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target, pos_weight=POS_WEIGHT)
    p = torch.sigmoid(logits)
    tp = (p * target).sum((1, 2, 3))
    fp = (p * (1 - target)).sum((1, 2, 3))
    fn = ((1 - p) * target).sum((1, 2, 3))
    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return bce + ((1 - tversky) ** gamma).mean()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tp = fp = fn = 0
    for x, y in loader:
        p = (torch.sigmoid(model(x.to(DEVICE))) > 0.5).float().cpu()
        tp += float((p * y).sum())
        fp += float((p * (1 - y)).sum())
        fn += float(((1 - p) * y).sum())
    return tp / (tp + fp + fn + 1e-6), 2 * tp / (2 * tp + fp + fn + 1e-6)


print("device:", DEVICE)

## Train

In [ ]:
model = UNet(in_ch=7).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), 2e-4, weight_decay=1e-4)
EPOCHS = 80
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
train_loader = DataLoader(ChangeDS("cd/train", augment=True, synthetic=True), batch_size=16, shuffle=True)
val_loader = DataLoader(ChangeDS("cd/val"), batch_size=8)
print("train", len(train_loader.dataset), "val", len(val_loader.dataset))

best = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    scheduler.step()
    iou, f1 = evaluate(model, val_loader)
    flag = ""
    if iou > best:
        best = iou
        torch.save(model.state_dict(), "change_model_improved.pt")
        flag = " <- best"
    if epoch % 5 == 0 or flag:
        print(f"epoch {epoch:2d} loss {total/len(train_loader.dataset):.4f} IoU {iou:.3f} F1 {f1:.3f}{flag}")
print(f"best change IoU {best:.3f}")

## Evaluate (strict and buffered)

In [ ]:
from skimage.morphology import binary_dilation, disk

model = UNet(in_ch=7).to(DEVICE)
model.load_state_dict(torch.load("change_model_improved.pt"))
model.eval()
iou, f1 = evaluate(model, DataLoader(ChangeDS("cd/val"), batch_size=8))
print(f"strict change IoU {iou:.3f}  F1 {f1:.3f}")

img0, H, W = rgb("dev", 2017)
img1, _, _ = rgb("dev", 2024)
o0, o1 = osm("dev", 2017), osm("dev", 2024)
h = min(img0.shape[1], img1.shape[1], o0.shape[0])
w = min(img0.shape[2], img1.shape[2], o0.shape[1])
img0, img1, o0, o1 = img0[:, :h, :w], img1[:, :h, :w], o0[:h, :w], o1[:h, :w]
truth = (o1 & ~o0).astype(bool)
full = np.concatenate([img0, img1, o0[None].astype("float32")], 0)

P, STR = 256, 192
prob = np.zeros((h, w), "float32")
cover = np.zeros((h, w), "float32")
ys = sorted(set(list(range(0, max(h - P, 0) + 1, STR)) + [h - P]))
xs = sorted(set(list(range(0, max(w - P, 0) + 1, STR)) + [w - P]))
with torch.no_grad():
    for y in ys:
        for x in xs:
            t = torch.from_numpy(full[:, y:y+P, x:x+P][None]).to(DEVICE)
            prob[y:y+P, x:x+P] += torch.sigmoid(model(t)).cpu().numpy()[0, 0]
            cover[y:y+P, x:x+P] += 1
pred = (prob / np.maximum(cover, 1)) > 0.5
for tol in (1, 2, 3):
    comp = (truth & binary_dilation(pred, disk(tol))).sum() / max(truth.sum(), 1)
    corr = (pred & binary_dilation(truth, disk(tol))).sum() / max(pred.sum(), 1)
    print(f"  +/-{tol}px buffered: completeness {comp:.3f} correctness {corr:.3f} F1 {2*comp*corr/(comp+corr+1e-9):.3f}")

## Change map

In [ ]:
import matplotlib.pyplot as plt

rgb_image = np.transpose(img1[:3], (1, 2, 0))
overlay = rgb_image.copy()
overlay[truth] = [0, 0.4, 1]
overlay[pred & ~truth] = [1, 1, 0]
overlay[pred & truth] = [1, 0, 0]
plt.figure(figsize=(11, 9))
plt.imshow(overlay)
plt.xticks([])
plt.yticks([])
plt.title("Change detection: new roads 2017->2024 (red=correct, yellow=false, blue=missed)")
plt.savefig("change_map_improved.png", dpi=140, bbox_inches="tight")
plt.show()